<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-19 16:27:00
=================== PORTFOLIO QUALITY ====================
qualified stocks: 84
with latest results: 84
still star stocks: 44
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.54 C
CAGR/XIRR %:     1.35%
Est FTT Value:   2.74 C
Future Upside:   1.20 C (77.71%)
=================== PERFORMANCE ==========================
Current PnL:             -41.25 L (-21.49%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    7.97 L (5.35%)
CY Booked PnL:           6.36 L (4.13%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               2.06 L (1.36%)
CY Investment:            1.92 C
Reserve:                  2.04 K
=================== P&L DETAIL ============================
Total Profit:             2.74 L
Total Loss:               -44.00 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
50,M&M,4.46,9.56,7.24,17.48,23881.0,28771.0,329846.0,3762.88,-2.93,73.0,X-LC,4.35,25.0,1.20,2.14,27.54,X40N,ATH,AUTO
75,TTKPRESTIG,0.52,-27.82,38.66,0.09,28123.0,-28033.0,72744.0,770.00,-26.87,52.0,M-SC,2.61,250.0,-1.00,0.47,30.93,OX40N,NTT,DURABLES
12,BOSCH-HCIL,1.41,-22.74,29.42,-0.01,29049.0,-29066.0,98739.0,2282.00,773.62,52.0,M-SC,18.83,231.0,-1.00,0.64,68.07,OX40N,BTT,AC
76,UNITDSPR,1.32,7.25,12.60,20.77,31641.0,16976.0,251120.0,1644.00,-6.03,63.0,X-MC,0.68,64.0,0.54,1.63,20.71,X40N,NTT,BREWERIES
52,MASFIN,-0.60,-19.93,51.44,21.26,40355.0,-19530.0,78450.0,396.02,-38.40,47.0,H-SC,14.50,167.0,-0.48,0.51,6.89,XR,ATH,FINANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
44,INFY,-1.52,-21.01,80.60,42.66,255221.0,-84204.0,316651.0,1972.00,-42.51,45.0,X-LC,6.62,3.0,-0.33,2.06,13.36,X40,NTT,IT
47,JIOFIN,1.02,-20.25,70.63,36.09,226187.0,-81303.0,320242.0,387.00,-29.17,56.0,X-LC,7.96,34.0,-0.36,2.08,10.90,XY24,BTT,FINANCE
38,HYUNDAI,5.80,13.48,25.39,42.29,81701.0,38213.0,321784.0,2782.73,32.44,67.0,X-LC,2.18,29.0,0.47,2.09,32.46,X40N,ATH,AUTO
33,HCLTECH,-0.88,-8.79,37.83,25.70,124546.0,-31747.0,329226.0,1837.07,-15.16,42.0,X-LC,9.36,8.0,-0.25,2.14,31.36,X40,ATH,IT
4,ASIANPAINT,0.57,-5.35,31.88,24.83,107547.0,-19067.0,337350.0,3422.40,-16.67,65.0,X-LC,2.97,30.0,-0.18,2.19,23.68,X40,ATH,PAINTS


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
81,VOLTAS,9.03,-3.23,50.04,45.20,92849.0,-6192.0,185550.0,1856.00,5.52,73.0,X-MC,16.17,79.0,-0.07,1.21,30.73,XY25,NTT,AC
51,MARUTI,7.58,0.00,25.98,25.99,88261.0,15.0,339725.0,17120.06,3.35,72.0,X-LC,8.86,33.0,0.00,2.21,21.66,XR,ATH,AUTO
38,HYUNDAI,5.80,13.48,25.39,42.29,81701.0,38213.0,321784.0,2782.73,32.44,67.0,X-LC,2.18,29.0,0.47,2.09,32.46,X40N,ATH,AUTO
32,HAVELLS,5.33,-23.12,70.94,31.42,178888.0,-75823.0,252168.0,2052.62,-34.20,66.0,X-MC,10.11,63.0,-0.42,1.64,19.07,X40,ATH,ELECTRICAL
19,COLPAL,5.22,-32.54,104.21,37.76,185139.0,-85705.0,177660.0,3628.04,-18.89,60.0,X-MC,14.33,62.0,-0.46,1.15,12.98,XY25,ATH,FMCG


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,-4.02,-24.28,184.17,115.17,377585.0,-65741.0,205020.0,1903.93,-17.74,59.0,H-SC,24.53,147.0,-0.17,1.33,81.24,AR,ATH,IT
78,VAIBHAVGBL,-2.53,-50.65,225.75,60.77,203640.0,-92569.0,90206.0,521.00,NaN,32.0,NaN,22.05,160.0,-0.45,0.59,3.99,XR,NTT,JEWELLERY
24,EASEMYTRIP,-1.88,-65.40,321.05,45.70,196428.0,-115632.0,61183.0,26.40,NaN,61.0,NaN,16.30,204.0,-0.59,0.40,17.64,XY24,NTT,TRAVEL
71,TCS,-1.55,-29.44,79.64,26.75,274262.0,-143698.0,344377.0,4208.34,-43.70,42.0,X-LC,7.28,2.0,-0.52,2.24,19.44,X40,ATH,IT
44,INFY,-1.52,-21.01,80.60,42.66,255221.0,-84204.0,316651.0,1972.00,-42.51,45.0,X-LC,6.62,3.0,-0.33,2.06,13.36,X40,NTT,IT


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.09,53.36,34.62,106.44,78387.0,78776.0,226420.0,1629.94,-4.52,47.0,M-SC,22.42,220.0,1.00,1.47,71.56,OX40N,ATH,CABLES
65,SIS,-0.93,-16.55,45.31,21.26,41982.0,-18378.0,92654.0,528.00,77.85,35.0,H-SC,5.46,168.0,-0.44,0.60,37.76,OX40N,NTT,MISC
12,BOSCH-HCIL,1.41,-22.74,29.42,-0.01,29049.0,-29066.0,98739.0,2282.00,773.62,52.0,M-SC,18.83,231.0,-1.00,0.64,68.07,OX40N,BTT,AC
75,TTKPRESTIG,0.52,-27.82,38.66,0.09,28123.0,-28033.0,72744.0,770.00,-26.87,52.0,M-SC,2.61,250.0,-1.00,0.47,30.93,OX40N,NTT,DURABLES
42,INDIGOPNTS,1.21,-28.78,40.43,0.02,50263.0,-50237.0,124322.0,1408.00,-32.02,54.0,M-SC,6.95,222.0,-1.00,0.81,41.65,OX40N,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.09,53.36,34.62,106.44,78387.0,78776.0,226420.0,1629.94,-4.52,47.0,M-SC,22.42,220.0,1.0,1.47,71.56,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.09,53.36,34.62,106.44,78387.0,78776.0,226420.0,1629.94,-4.52,47.0,M-SC,22.42,220.0,1.00,1.47,71.56,OX40N,ATH,CABLES
66,STARHEALTH,1.58,10.04,33.83,47.26,104652.0,28223.0,309346.0,761.00,-19.51,57.0,H-SC,3.60,173.0,0.27,2.01,33.75,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.09,53.36,34.62,106.44,78387.0,78776.0,226420.0,1629.94,-4.52,47.0,M-SC,22.42,220.0,1.00,1.47,71.56,OX40N,ATH,CABLES
83,WIPRO,-0.65,-15.47,128.35,93.01,271486.0,-38720.0,211520.0,420.00,-27.55,45.0,M-LC,8.21,98.0,-0.14,1.37,17.31,XR,NTT,IT
69,TANLA,-4.02,-24.28,184.17,115.17,377585.0,-65741.0,205020.0,1903.93,-17.74,59.0,H-SC,24.53,147.0,-0.17,1.33,81.24,AR,ATH,IT
41,INDIAMART,0.21,-33.29,201.17,100.92,165527.0,-41054.0,82282.0,4675.60,-60.27,46.0,H-SC,16.59,140.0,-0.25,0.53,4.39,AR,ATH,MISC
3,ALKYLAMINE,-0.79,-33.02,187.44,92.54,126766.0,-33333.0,67630.0,4520.85,-43.63,37.0,H-SC,8.06,142.0,-0.26,0.44,28.42,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
54,PAGEIND,0.83,-18.97,63.18,32.21,137388.0,-50924.0,217455.0,50690.44,-38.50,31.0,X-SC,6.16,55.0,-0.37,1.41,4.34,X40,ATH,APPARELS
37,HONAUT,0.57,-22.20,93.94,50.88,140933.0,-42813.0,150025.0,58192.80,-39.09,40.0,X-SC,8.98,90.0,-0.30,0.97,14.30,X40N,ATH,ELECTRICAL
73,TMPV,4.34,-34.30,127.70,49.60,400087.0,-163570.0,313302.0,600.00,-533.92,41.0,X-MC,17.15,4.0,-0.41,2.03,4.34,XY24,NTT,AUTO
33,HCLTECH,-0.88,-8.79,37.83,25.70,124546.0,-31747.0,329226.0,1837.07,-15.16,42.0,X-LC,9.36,8.0,-0.25,2.14,31.36,X40,ATH,IT
71,TCS,-1.55,-29.44,79.64,26.75,274262.0,-143698.0,344377.0,4208.34,-43.70,42.0,X-LC,7.28,2.0,-0.52,2.24,19.44,X40,ATH,IT


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
76,UNITDSPR,1.32,7.25,12.60,20.77,31641.0,16976.0,251120.0,1644.00,-6.03,63.0,X-MC,0.68,64.0,0.54,1.63,20.71,X40N,NTT,BREWERIES
1,ABBOTINDIA,0.88,-13.80,33.84,15.37,52171.0,-24685.0,154170.0,34390.26,-30.90,46.0,X-MC,0.80,58.0,-0.47,1.00,4.43,X40,ATH,PHARMA
17,CIPLA,0.11,6.63,16.19,23.89,51902.0,19926.0,320581.0,1648.14,40.23,63.0,X-MC,1.91,18.0,0.38,2.08,20.08,X40N,ATH,PHARMA
38,HYUNDAI,5.80,13.48,25.39,42.29,81701.0,38213.0,321784.0,2782.73,32.44,67.0,X-LC,2.18,29.0,0.47,2.09,32.46,X40N,ATH,AUTO
11,BERGEPAINT,-0.22,-14.10,45.89,25.32,101600.0,-36351.0,221398.0,680.00,-23.24,46.0,X-MC,2.53,68.0,-0.36,1.44,16.10,XY24,NTT,PAINTS


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
26,GILLETTE,-1.36,-15.48,52.25,28.69,112131.0,-39291.0,214605.0,10891.01,-39.47,47.0,X-SC,6.13,83.0,-0.35,1.39,3.69,X40,ATH,FMCG
55,PGHH,1.36,-48.13,152.59,31.02,168996.0,-102768.0,110752.0,17484.41,-61.07,46.0,X-SC,22.56,57.0,-0.61,0.72,4.26,X40,ATH,FMCG
73,TMPV,4.34,-34.30,127.70,49.60,400087.0,-163570.0,313302.0,600.00,-533.92,41.0,X-MC,17.15,4.0,-0.41,2.03,4.34,XY24,NTT,AUTO
54,PAGEIND,0.83,-18.97,63.18,32.21,137388.0,-50924.0,217455.0,50690.44,-38.50,31.0,X-SC,6.16,55.0,-0.37,1.41,4.34,X40,ATH,APPARELS
1,ABBOTINDIA,0.88,-13.80,33.84,15.37,52171.0,-24685.0,154170.0,34390.26,-30.90,46.0,X-MC,0.80,58.0,-0.47,1.00,4.43,X40,ATH,PHARMA


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
10,BATAINDIA,0.51,-59.54,234.18,35.20,121909.0,-76612.0,52058.0,2096.00,-19.19,45.0,X-SC,6.52,91.0,-0.63,0.34,8.18,X40,NTT,FOOTWEAR
55,PGHH,1.36,-48.13,152.59,31.02,168996.0,-102768.0,110752.0,17484.41,-61.07,46.0,X-SC,22.56,57.0,-0.61,0.72,4.26,X40,ATH,FMCG
37,HONAUT,0.57,-22.20,93.94,50.88,140933.0,-42813.0,150025.0,58192.80,-39.09,40.0,X-SC,8.98,90.0,-0.30,0.97,14.30,X40N,ATH,ELECTRICAL
2,ACC,3.15,-43.22,207.90,74.81,316490.0,-115896.0,152232.0,3906.00,21.51,64.0,X-SC,11.60,85.0,-0.37,0.99,12.77,XY24,BTT,CEMENT
1,ABBOTINDIA,0.88,-13.80,33.84,15.37,52171.0,-24685.0,154170.0,34390.26,-30.90,46.0,X-MC,0.80,58.0,-0.47,1.00,4.43,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,1.48,-14.29,50.17,28.70,188161.0,-62549.0,375046.0,452.00,-18.77,75.0,X-LC,5.77,1.0,-0.33,2.44,18.04,X40,NTT,FMCG
71,TCS,-1.55,-29.44,79.64,26.75,274262.0,-143698.0,344377.0,4208.34,-43.70,42.0,X-LC,7.28,2.0,-0.52,2.24,19.44,X40,ATH,IT
44,INFY,-1.52,-21.01,80.60,42.66,255221.0,-84204.0,316651.0,1972.00,-42.51,45.0,X-LC,6.62,3.0,-0.33,2.06,13.36,X40,NTT,IT
73,TMPV,4.34,-34.30,127.70,49.60,400087.0,-163570.0,313302.0,600.00,-533.92,41.0,X-MC,17.15,4.0,-0.41,2.03,4.34,XY24,NTT,AUTO
33,HCLTECH,-0.88,-8.79,37.83,25.70,124546.0,-31747.0,329226.0,1837.07,-15.16,42.0,X-LC,9.36,8.0,-0.25,2.14,31.36,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,1.41,-22.74,29.42,-0.01,29049.0,-29066.0,98739.0,2282.00,773.62,52.0,M-SC,18.83,231.0,-1.00,0.64,68.07,OX40N,BTT,AC
79,VALIANTORG,-0.92,-29.32,146.40,74.15,169288.0,-47971.0,115634.0,838.00,NaN,37.0,NaN,34.99,145.0,-0.28,0.75,72.08,XR,NTT,CHEMICALS
69,TANLA,-4.02,-24.28,184.17,115.17,377585.0,-65741.0,205020.0,1903.93,-17.74,59.0,H-SC,24.53,147.0,-0.17,1.33,81.24,AR,ATH,IT
27,GLAXO,2.85,8.93,24.57,35.69,53335.0,17788.0,217072.0,3380.16,-20.88,58.0,X-MC,6.48,56.0,0.33,1.41,29.01,X40,ATH,PHARMA
25,FINCABLES,2.09,53.36,34.62,106.44,78387.0,78776.0,226420.0,1629.94,-4.52,47.0,M-SC,22.42,220.0,1.00,1.47,71.56,OX40N,ATH,CABLES


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,-4.02,-24.28,184.17,115.17,377585.0,-65741.0,205020.0,1903.93,-17.74,59.0,H-SC,24.53,147.0,-0.17,1.33,81.24,AR,ATH,IT
81,VOLTAS,9.03,-3.23,50.04,45.20,92849.0,-6192.0,185550.0,1856.00,5.52,73.0,X-MC,16.17,79.0,-0.07,1.21,30.73,XY25,NTT,AC
50,M&M,4.46,9.56,7.24,17.48,23881.0,28771.0,329846.0,3762.88,-2.93,73.0,X-LC,4.35,25.0,1.20,2.14,27.54,X40N,ATH,AUTO
51,MARUTI,7.58,0.00,25.98,25.99,88261.0,15.0,339725.0,17120.06,3.35,72.0,X-LC,8.86,33.0,0.00,2.21,21.66,XR,ATH,AUTO
61,REPCOHOME,-0.45,-18.10,131.76,89.82,309181.0,-51850.0,234655.0,880.00,NaN,68.0,NaN,8.13,161.0,-0.17,1.52,26.67,XY24,NTT,FINANCE


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.77
1,25,51.72
2,50,82.49


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    39.04
SC    28.34
MC    21.67
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.00
XY24     22.78
X40N     19.47
XY25     10.24
XR        9.88
OX40N     6.68
AR        3.20
SR        0.72
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    37.67
X-MC    20.26
H-SC    14.02
X-SC     8.71
M-SC     5.61
M-LC     1.37
M-MC     1.07
L-MC     0.34
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,18.65,-22.60,66.82
IT,12.29,-35.56,106.18
FINANCE,10.94,-20.93,72.39
AUTO,9.35,-10.94,51.70
PAINTS,5.59,-23.14,45.26
MISC,5.54,-44.41,102.45
INSURANCE,5.41,0.91,36.33
BANKS,5.29,-9.94,65.07
PHARMA,4.49,1.88,22.75


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,3985242.0,19
X40,2173337.0,16
XR,1584163.0,13
X40N,1434103.0,12
AR,1013784.0,6
OX40N,759681.0,10
XY25,713259.0,6
SR,303147.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2448982.0,17
H-SC,2284844.0,15
X-MC,1845492.0,14
X-SC,1460853.0,8
M-SC,508301.0,7
M-LC,271486.0,1
M-MC,80844.0,1
L-MC,49952.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1290564.0      8
H-SC       XY24       987608.0      5
X-SC       XY24       701533.0      2
X-MC       XY24       668700.0      3
           X40N       556455.0      5
H-SC       AR         543112.0      2
X-SC       X40        540424.0      4
X-LC       X40N       489543.0      4
H-SC       XR         377195.0      4
X-LC       XY25       354427.0      3
X-MC       X40        342349.0      4
           XY25       277988.0      2
M-LC       XR         271486.0      1
H-SC       OX40N      250163.0      3
X-LC       XY24       226187.0      1
X-SC       X40N       218896.0      2
M-SC       XR         193096.0      2
           OX40N      185822.0      4
           XY24       129383.0      1
H-SC       SR         126766.0      1
X-LC       XR          88261.0      1
M-MC       XY25        80844.0      1
L-MC       XR          49952.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 36.0 seconds
